[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_computing/04_vectorization_and_numpy_performance/exercises.ipynb)

# Exercises — Topic 04: Vectorization and NumPy Performance

20 fully solved problems in 4 levels. Attempt each before opening its solution cell.

Machine model used throughout unless stated otherwise: CPU with $P_{\text{peak}} = 2$ Tflop/s (binary64), $B_{\text{mem}} = 200$ GB/s, cache line $64$ B, L1 $= 32$ KiB, L2 $= 1$ MiB, L3 $= 32$ MiB. GPU examples use an A100: $P_{\text{peak}} = 312$ Tflop/s (fp16 tensor core), $B_{\text{mem}} = 1.55$ TB/s, SRAM $\approx 192$ KiB per SM.

## Level 0 — Concept Check

### Problem L0.1: View or copy?

For a C-contiguous `X` of shape $(1000, 1000)$, dtype float64, classify each as a **view** (metadata only) or a **copy** (full data pass), and state the cost:

(a) `X.T` (b) `X[::2, :]` (c) `X[[0, 5, 9], :]` (d) `X.reshape(1000000)` (e) `X.T.reshape(1000000)` (f) `X.astype(np.float32)` (g) `np.broadcast_to(X[0], (1000, 1000))`

**Solution**

| Expression | Result | Reason | Cost |
|---|---|---|---|
| (a) `X.T` | **view** | strides reversed: $(8, 8000)$ | $O(1)$ |
| (b) `X[::2, :]` | **view** | stride doubled on axis 0: $(16000, 8)$ | $O(1)$ |
| (c) `X[[0,5,9], :]` | **copy** | fancy indexing gathers non-uniform offsets, inexpressible as a stride | $O(3 \times 1000)$ |
| (d) `X.reshape(10^{6})` | **view** | C-contiguous, so the flat traversal already matches | $O(1)$ |
| (e) `X.T.reshape(10^{6})` | **copy** | `X.T` is F-contiguous; flattening in C order needs a physical transpose | $O(10^{6})$ |
| (f) `X.astype(np.float32)` | **copy** | itemsize changes, new buffer required | $O(10^{6})$ |
| (g) `np.broadcast_to(X[0], (1000,1000))` | **view** | axis 0 given stride $0$ | $O(1)$ |

The single decision rule:

$$
\boxed{\text{view} \iff \text{the requested access pattern is expressible as } \mathrm{offset}(i) = \textstyle\sum_k i_k s_k \text{ on the same buffer}}
$$

*Key takeaway*: check with `Y.base is not None` and `np.shares_memory(X, Y)`. Each unexpected copy is one full sweep of memory — for a memory-bound pipeline, that is one full unit of runtime.

### Problem L0.2: Strides arithmetic

An array has shape $(4, 3, 5)$, dtype float64, C order. (a) Give its strides. (b) At what byte offset does element $(2, 1, 3)$ live? (c) Give the strides after `.transpose(2, 0, 1)`. (d) Is the transposed array C-contiguous, F-contiguous, or neither?

**Solution**

**(a)** C order: the last axis is fastest, and $s_k = s_{k+1}n_{k+1}$ with itemsize $8$:

$$
s_3 = 8, \qquad s_2 = 8 \times 5 = 40, \qquad s_1 = 40 \times 3 = 120
$$

so $\text{strides} = (120, 40, 8)$ bytes.

**(b)**

$$
\mathrm{offset}(2, 1, 3) = 2 \times 120 + 1 \times 40 + 3 \times 8 = 240 + 40 + 24 = 304 \text{ bytes}
$$

(Equivalently element index $304/8 = 38 = 2\cdot15 + 1\cdot5 + 3$ ✓.)

**(c)** `transpose(2, 0, 1)` permutes both shape and strides by the same permutation: shape $(5, 4, 3)$, strides $(8, 120, 40)$.

**(d)** Neither. C-contiguity for shape $(5,4,3)$ would require strides $(96, 24, 8)$; F-contiguity would require $(8, 40, 160)$. The array is a valid strided view but has no contiguous flag, so a subsequent `reshape` or BLAS call must copy.

$$
\boxed{(120, 40, 8); \quad \mathrm{offset} = 304 \text{ B}; \quad \text{after transpose } (8, 120, 40), \text{ neither contiguous}}
$$

*Key takeaway*: transposition permutes the strides tuple — free. "Neither contiguous" is the flag that predicts a hidden copy downstream.

### Problem L0.3: Broadcast or error?

For each pair of shapes, give the broadcast result or explain the failure: (a) $(5, 1, 4)$ and $(3, 1)$; (b) $(8, 1, 6, 1)$ and $(7, 1, 5)$; (c) $(3, 4)$ and $(4, 3)$; (d) $(2, 3)$ and $(2, 1, 3)$; (e) $(0, 4)$ and $(4,)$.

**Solution**

Right-align, pad with $1$s, then each axis must be equal or contain a $1$.

**(a)** $(5,1,4)$ vs $(1,3,1)$ $\Rightarrow$ $(5, 3, 4)$. ✓

**(b)** $(8,1,6,1)$ vs $(1,7,1,5)$ $\Rightarrow$ $(8, 7, 6, 5)$. ✓ — the textbook four-axis case.

**(c)** $(3,4)$ vs $(4,3)$: axis $-1$ has $4$ vs $3$ (neither is $1$) $\Rightarrow$ **error**. This is the most common broadcasting bug: a missing transpose.

**(d)** $(1,2,3)$ vs $(2,1,3)$ $\Rightarrow$ $(2, 2, 3)$. ✓ — note this *silently* produces an outer-product-like result; if you meant elementwise, the shapes were wrong.

**(e)** $(0,4)$ vs $(1,4)$ $\Rightarrow$ $(0, 4)$. ✓ — zero-size arrays broadcast fine and produce empty results; a frequent source of silent no-ops.

$$
\boxed{\text{(a) } (5,3,4), \text{ (b) } (8,7,6,5), \text{ (c) error, (d) } (2,2,3), \text{ (e) } (0,4)}
$$

*Key takeaway*: case (d) is the dangerous one — legal, silent, and $O(n^{2})$ memory instead of $O(n)$. Assert output shapes in numerical code.

### Problem L0.4: Which side of the roofline?

Classify each kernel as memory-bound or compute-bound on the CPU model ($I^{*} = 10$ flop/byte), computing $I$ for binary64 data: (a) `y = a*x + y` on $n = 10^{7}$; (b) `C = A @ B` with $n = 2000$; (c) `s = (x**2).sum()`; (d) elementwise `np.exp(x)`; (e) `A @ b` (matrix–vector, $n = 5000$).

**Solution**

**(a) `axpy`.** $2n$ flops; $24n$ bytes (read $x$, read $y$, write $y$): $I = \frac{1}{12} \approx 0.083$. **Memory-bound** by a factor $120$.

**(b) `gemm`.** $2n^{3}$ flops; $\ge 24n^{2}$ compulsory bytes: $I = n/12 \approx 167$. **Compute-bound** (well past $I^{*} = 10$).

**(c) `(x**2).sum()`.** As written, `x**2` materializes a temporary: $8n$ read $+\,8n$ write, then $8n$ read for the sum $= 24n$ bytes for $2n$ flops, $I = \frac{1}{12}$. Fused (`np.dot(x, x)` or `np.einsum('i,i->', x, x)`): $8n$ bytes, $2n$ flops, $I = \frac{1}{4}$ — a $3\times$ traffic reduction for free. Still **memory-bound**.

**(d) `np.exp(x)`.** Roughly $20$ flops per element via polynomial approximation; $16n$ bytes: $I \approx 1.25$. **Memory-bound** — transcendental functions are cheaper than the memory that feeds them, which is why fusing `exp` into a neighbouring kernel is nearly free.

**(e) `gemv`.** $2n^{2}$ flops; $8n^{2}$ bytes: $I = \frac{1}{4}$, independent of $n$. **Memory-bound forever** — no cache blocking can help because each entry of $A$ is used exactly once.

$$
\boxed{\text{(a) } \tfrac{1}{12}, \text{ (c) } \tfrac{1}{12} \to \tfrac{1}{4}, \text{ (d) } 1.25, \text{ (e) } \tfrac{1}{4} \text{ — all memory-bound; (b) } 167 \text{ — compute-bound}}
$$

*Key takeaway*: only level-3 operations live on the compute side. Everything elementwise, every reduction, and every matrix–vector product runs at the speed of memory.

## Level 1 — Foundation

### Problem L1.1: Predicting runtimes from the roofline

On the CPU model, predict the runtime of (a) `axpy` with $n = 10^{8}$ float64; (b) `gemv` with $n = 10^{4}$; (c) `gemm` with $n = 10^{3}$. Then compute the achieved fraction of peak flops in each case.

**Solution**

Use $T \ge \max\left( \frac{\text{flops}}{P_{\text{peak}}}, \frac{\text{bytes}}{B_{\text{mem}}} \right)$.

**(a) `axpy`, $n = 10^{8}$.** Flops $= 2 \times 10^{8}$; bytes $= 24 \times 10^{8} = 2.4$ GB.

$$
T_{\text{mem}} = \frac{2.4 \times 10^{9}}{2 \times 10^{11}} = 12 \text{ ms}, \qquad T_{\text{flop}} = \frac{2 \times 10^{8}}{2 \times 10^{12}} = 0.1 \text{ ms}
$$

$T \approx 12$ ms; achieved rate $= 2\times10^{8}/0.012 = 16.7$ Gflop/s $= \mathbf{0.83\%}$ of peak.

**(b) `gemv`, $n = 10^{4}$.** Flops $= 2 \times 10^{8}$; bytes $= 8 \times 10^{8} = 0.8$ GB.

$$
T = \frac{8 \times 10^{8}}{2 \times 10^{11}} = 4 \text{ ms}, \qquad \text{rate} = 50 \text{ Gflop/s} = \mathbf{2.5\%} \text{ of peak}
$$

**(c) `gemm`, $n = 10^{3}$.** Flops $= 2 \times 10^{9}$; bytes $\ge 24 \times 10^{6} = 24$ MB.

$$
T_{\text{flop}} = \frac{2 \times 10^{9}}{2 \times 10^{12}} = 1 \text{ ms}, \qquad T_{\text{mem}} = \frac{2.4\times10^{7}}{2\times10^{11}} = 0.12 \text{ ms}
$$

$T \approx 1$ ms; rate $= 2$ Tflop/s $= \mathbf{100\%}$ of peak (in the ideal model; $80$–$90\%$ in practice).

$$
\boxed{\texttt{axpy}: 12\,\text{ms} \; (0.8\%), \quad \texttt{gemv}: 4\,\text{ms} \; (2.5\%), \quad \texttt{gemm}: 1\,\text{ms} \; (100\%)}
$$

*Key takeaway*: the `gemm` performs $10\times$ the flops of the `axpy` in $\frac{1}{12}$ of the time. Flop counts predict nothing across roofline regimes.

### Problem L1.2: The cost of traversal order

An $8192 \times 8192$ float64 array ($512$ MiB) is summed (a) along axis 1 and (b) along axis 0 with a hand-written loop that accumulates one element at a time. Count the bytes actually transferred in each case and predict the ratio. (c) Why does `X.sum(axis=0)` in NumPy *not* show this penalty?

**Solution**

**(a) Axis-1 traversal (row-wise, C order).** Stride is $8$ B, so each $64$ B cache line supplies $8$ useful elements:

$$
\text{bytes} = 8 \times 8192^{2} = 5.37 \times 10^{8} \approx 512 \text{ MiB}, \qquad T = \frac{5.37\times10^{8}}{2\times10^{11}} = 2.7 \text{ ms}
$$

**(b) Axis-0 traversal (column-wise).** Stride is $8 \times 8192 = 65536$ B $\gg 64$ B, so every access pulls a fresh line and uses $8$ of its $64$ bytes:

$$
\text{bytes} = 64 \times 8192^{2} = 4.29 \times 10^{9} \approx 4 \text{ GiB}, \qquad T = 21.5 \text{ ms}
$$

$$
\text{ratio} = \frac{64}{8} = 8\times
$$

and in practice worse: with $8192$ columns each $64$ KiB apart, the working set of one "row sweep" is $8192$ distinct pages, exceeding typical TLB reach ($\sim 1500$ entries), so each access may also cost a page-table walk. Measured ratios of $15$–$30\times$ are normal.

**(c)** NumPy's reduction does not iterate in the mathematical order. For `sum(axis=0)` it loops over rows in the outer loop and accumulates into a *contiguous output vector* of length $8192$ ($64$ KiB, L2-resident) in the inner loop. Every memory access — both the input sweep and the accumulator — is then contiguous, and the traffic returns to $512$ MiB. **The layout was never the problem; the loop nest was.**

$$
\boxed{\text{hand loop: } 512\,\text{MiB vs } 4\,\text{GiB} \; (8\times, \text{ often } 20\times); \text{ NumPy reorders loops and pays neither}}
$$

*Key takeaway*: layout matters exactly when the loop order is fixed — in your own Numba/Cython kernels, in fancy indexing, and in ufuncs on mismatched layouts.

### Problem L1.3: Counting temporaries

Consider `z = a*x**2 + b*x + c` where `x` is $n = 10^{8}$ float64 and `a, b, c` are scalars. (a) Count the temporaries and total memory traffic. (b) Rewrite with Horner's rule and in-place operations; recount. (c) Predict the speedup and give the exact NumPy code.

**Solution**

**(a)** Evaluating left to right, NumPy allocates one array per binary ufunc node: `x**2` ($t_1$), `a*t_1` ($t_2$), `b*x` ($t_3$), `t_2 + t_3` ($t_4$), `t_4 + c` ($t_5$). That is 5 array-sized passes writing plus their reads. Counting read+write per node ($2$ ops touching $N$ elements each, one write):

$$
\text{traffic} \approx (2 \times 5 + 1) \times 8n = 88n \text{ bytes} = 8.8 \text{ GB}, \qquad T \approx \frac{8.8\times10^{9}}{2\times10^{11}} = 44 \text{ ms}
$$

Peak extra memory: up to 3 live temporaries $= 2.4$ GB.

**(b) Horner + in-place.** $z = (a x + b)x + c$ needs 2 multiplies and 2 adds, and can run in a single preallocated buffer:

```python
z = np.empty_like(x)
np.multiply(x, a, out=z)     # z = a*x
np.add(z, b, out=z)          # z = a*x + b
np.multiply(z, x, out=z)     # z = (a*x + b)*x
np.add(z, c, out=z)          # z = (a*x + b)*x + c
```

Traffic: pass 1 reads $x$, writes $z$ ($16n$); passes 2 and 4 read and write $z$ ($16n$ each); pass 3 reads $z$ and $x$, writes $z$ ($24n$). Total $72n$ — better, but still 4 sweeps. A **fused** kernel (Numba/numexpr) reads $x$ once and writes $z$ once:

$$
\text{traffic}_{\text{fused}} = 16n = 1.6 \text{ GB}, \qquad T \approx 8 \text{ ms}
$$

**(c)** Predicted speedup $\approx 88/16 = 5.5\times$, with zero temporaries and a lower flop count (Horner: $4n$ flops vs $5n$ for the naive form). In practice `numexpr.evaluate("a*x**2 + b*x + c")` achieves most of this automatically and threads it as well.

$$
\boxed{88n \text{ bytes} \to 72n \text{ (in-place)} \to 16n \text{ (fused)}: \approx 5.5\times \text{ speedup, memory-bound throughout}}
$$

*Key takeaway*: for memory-bound expressions, count **sweeps over the array**, not flops. Each binary ufunc node is one sweep.

### Problem L1.4: Optimal block size for each cache level

For binary64 blocked matrix multiplication with the capacity constraint $3b^{2} \le M$: (a) compute $b$ for L1 ($32$ KiB), L2 ($1$ MiB), L3 ($32$ MiB); (b) give the resulting arithmetic intensity of the L1 micro-kernel; (c) explain why real BLAS blocks for all three levels simultaneously.

**Solution**

**(a)** $M$ in binary64 words is (cache bytes)/8, and $b = \lfloor\sqrt{M/3}\rfloor$:

| Level | Bytes | $M$ (words) | $M/3$ | $b$ |
|---|---|---|---|---|
| L1 | $32$ KiB | $4096$ | $1365$ | $36$ |
| L2 | $1$ MiB | $131072$ | $43691$ | $209$ |
| L3 | $32$ MiB | $4194304$ | $1398101$ | $1182$ |

**(b)** A $b \times b$ tile product performs $2b^{3}$ flops on $3b^{2}$ words $= 24b^{2}$ bytes:

$$
I = \frac{2b^{3}}{24b^{2}} = \frac{b}{12} = \frac{36}{12} = 3 \; \frac{\text{flop}}{\text{byte}}
$$

with respect to L2. That is below the DRAM ridge point $I^{*} = 10$ — which is exactly why one cache level is not enough.

**(c)** Each level has a different capacity and bandwidth, so each needs its own reuse factor. Blocking for L3 ($b \approx 1182$, $I \approx 98$ against DRAM) makes DRAM traffic negligible; blocking for L2 within that makes L3 traffic negligible; the L1/register micro-kernel ($m_r \times n_r$ accumulators held in vector registers, typically $6 \times 8$ or $8 \times 8$) keeps the ALUs saturated. The composite reuse is the *product* of the per-level factors, and only then does the whole hierarchy stay ahead of the FMA units. Goto's algorithm additionally *packs* the $A$- and $B$-panels into contiguous scratch buffers so that the micro-kernel's loads are unit-stride and TLB-friendly.

$$
\boxed{b_{\text{L1}} = 36, \; b_{\text{L2}} = 209, \; b_{\text{L3}} = 1182; \quad I_{\text{tile}} = b/12}
$$

*Key takeaway*: $I = \Theta(\sqrt{M})$ — arithmetic intensity is bought with cache capacity, and the square root is why caches must grow quadratically to keep up with flop-rate growth.

### Problem L1.5: Broadcasting's hidden memory bill

You compute `D = ((A[:, None, :] - B[None, :, :])**2).sum(-1)` with $A \in \mathbb{R}^{m \times d}$, $B \in \mathbb{R}^{n \times d}$, float64, $m = n = 4000$, $d = 128$. (a) Size of the largest temporary. (b) Total traffic and arithmetic intensity. (c) Same for the `gemm` formulation. (d) The accuracy caveat.

**Solution**

**(a)** The difference has shape $(m, n, d) = (4000, 4000, 128)$:

$$
8 \times 4000 \times 4000 \times 128 = 1.64 \times 10^{11} \text{ bytes} = 164 \text{ GB}
$$

On most machines this raises `MemoryError` outright — the algorithm is not merely slow, it is infeasible.

**(b)** Traffic: write the difference, read it for the square, write the square, read it for the reduction $\approx 4 \times 8mnd = 6.6 \times 10^{11}$ bytes. Flops $\approx 3mnd = 6.1 \times 10^{9}$:

$$
I = \frac{3mnd}{32mnd} = 0.094, \qquad T \ge \frac{6.6\times10^{11}}{2\times10^{11}} = 3.3 \text{ s}
$$

**(c) Gram identity.** $D = \Vert a \Vert^{2}\mathbf{1}^{\top} + \mathbf{1}\Vert b \Vert^{2\top} - 2AB^{\top}$. Flops $= 2mnd = 4.1 \times 10^{9}$ (dominated by the `gemm`); compulsory bytes $= 8(md + nd + mn) = 8(5.1\times10^{5} + 5.1\times10^{5} + 1.6\times10^{7}) = 1.4 \times 10^{8}$:

$$
I = \frac{4.1\times10^{9}}{1.4\times10^{8}} \approx 29 \; \gt \; I^{*} = 10 \quad \Rightarrow \quad T \approx \frac{4.1\times10^{9}}{2\times10^{12}} = 2.1 \text{ ms}
$$

A speedup of $\approx 1500\times$, and peak memory drops from $164$ GB to $128$ MB.

**(d)** The identity computes $\Vert a \Vert^{2} + \Vert b \Vert^{2} - 2a^{\top}b$. When $a \approx b$ the three terms are nearly equal and the result is a small difference of large quantities — Topic 02's catastrophic cancellation, with amplification factor $\approx \frac{2\Vert a \Vert^{2}}{\Vert a - b \Vert^{2}}$. Computed self-distances can come out slightly *negative*; the standard mitigations are `np.maximum(D, 0)` before `sqrt`, setting the diagonal to zero exactly, and using the direct formulation when small distances must be accurate (e.g. clustering with tight clusters).

$$
\boxed{\text{broadcast: } 164\,\text{GB temp}, I = 0.094, \; 3.3\,\text{s}; \quad \texttt{gemm}: 128\,\text{MB}, I = 29, \; 2.1\,\text{ms}}
$$

*Key takeaway*: broadcasting is free on inputs and fully priced on outputs. Whenever a broadcast is immediately reduced, look for the algebraic identity that turns it into a matrix product.

### Problem L1.6: dtype as a performance knob

A pipeline processes $10^{9}$ elements through several elementwise stages. (a) Predict the speedup of float32 over float64 for a memory-bound stage. (b) For a compute-bound stage on a SIMD CPU. (c) What accuracy question must be answered first, and with which tool?

**Solution**

**(a) Memory-bound.** Runtime $= \text{bytes}/B_{\text{mem}}$, and bytes halve:

$$
\frac{T_{64}}{T_{32}} = \frac{8n \cdot k}{4n \cdot k} = 2.0\times
$$

exactly, with no dependence on the arithmetic performed. Storage and cache residency also halve, which can additionally move a working set below a cache threshold — producing superlinear jumps.

**(b) Compute-bound.** An AVX-512 register holds $8$ binary64 or $16$ binary32 lanes, so the vector throughput doubles:

$$
\frac{T_{64}}{T_{32}} \approx 2.0\times
$$

again — but only for kernels that vectorize. On GPUs the ratio is hardware-specific: consumer cards often have fp64 at $\frac{1}{32}$ of fp32 (a $32\times$ gap), while data-center parts run $\frac{1}{2}$; tensor cores add another $8$–$16\times$ for fp16/bf16.

**(c)** Before halving precision, apply **Topic 03's digit rule**: correct digits $\approx p - \log_{10}\kappa$, with $p \approx 16$ for fp64 and $p \approx 7.2$ for fp32. So fp32 is safe when

$$
\log_{10}\kappa \; \lesssim \; 7.2 - (\text{digits required})
$$

For a $\kappa = 10^{4}$ problem needing 3 digits: $7.2 - 4 = 3.2 \ge 3$ ✓. For $\kappa = 10^{6}$: $7.2 - 6 = 1.2 \lt 3$ ✗ — keep fp64, or split the computation so that only the well-conditioned parts run narrow (mixed precision, Topic 05).

$$
\boxed{2\times \text{ in both regimes (CPU); permissible iff } \log_{10}\kappa \lesssim 7.2 - d_{\text{required}}}
$$

*Key takeaway*: dtype is the one knob that is simultaneously a performance decision and an accuracy decision — always price both.

## Level 2 — Applications in AI/ML

### Problem L2.1: Embedding lookup versus one-hot matmul

A vocabulary of $V = 50000$ tokens is embedded into $H = 768$ dimensions for a batch of $n = 1024$ tokens. Compare (a) one-hot matrix $X \in \{0,1\}^{n \times V}$ times $W \in \mathbb{R}^{V \times H}$ with (b) `W[idx]`. Give flops, bytes, and the ratio. (c) What does the backward pass look like in each case?

**Solution**

**(a) One-hot matmul.** Flops:

$$
2nVH = 2 \times 1024 \times 50000 \times 768 = 7.9 \times 10^{10}
$$

of which all but $2nH = 1.6 \times 10^{6}$ are multiplications by zero — a useful-work fraction of $2 \times 10^{-5}$. Bytes: the one-hot matrix alone is $4nV = 2.05 \times 10^{8}$ bytes (fp32), plus $4VH = 1.5 \times 10^{8}$ for $W$.

**(b) Fancy-index lookup `W[idx]`.** No flops at all — it is a gather of $n$ rows:

$$
\text{bytes} = 4nH \times 2 = 6.3 \times 10^{6} \; (\text{read} + \text{write})
$$

**Ratio.** Flops: $7.9\times10^{10}$ vs $0$. Traffic: $3.5\times10^{8}$ vs $6.3\times10^{6}$, a $56\times$ reduction. Wall-clock on a GPU: milliseconds versus microseconds.

**(c) Backward.** For (a) the gradient is $\nabla_W = X^{\top}\nabla_Y$ — another $2nVH$ flops producing a *dense* $V \times H$ gradient. For (b) it is a **scatter-add**: `np.add.at(gradW, idx, gradY)` (or a sparse gradient), touching only the $\le n$ rows actually used. The sparse gradient is why optimizers offer `sparse=True` embedding modes; note that Adam's per-parameter state updates must then also be restricted to the touched rows, or the optimizer silently becomes the dominant cost.

$$
\boxed{\text{one-hot: } 7.9\times10^{10} \text{ flops}, \; 350\,\text{MB}; \quad \text{lookup: } 0 \text{ flops}, \; 6.3\,\text{MB} \; (56\times \text{ less traffic})}
$$

*Key takeaway*: whenever a matrix is structurally sparse (one-hot, permutation, diagonal, banded), the fastest kernel is an *indexing* operation, not a multiplication.

### Problem L2.2: LayerNorm — three passes or one?

LayerNorm normalizes over the last axis of a $(B, T, H) = (32, 512, 1024)$ float32 tensor. (a) Count traffic for a naive three-pass implementation (mean, variance, normalize). (b) For a single-pass Welford implementation. (c) Predict the speedup on the CPU model, and state the extra benefit.

**Solution**

Tensor size: $N = 32 \times 512 \times 1024 = 1.68 \times 10^{7}$ elements $= 67$ MB (fp32) — far larger than L3, so every pass is a DRAM round trip.

**(a) Three passes.**
1. Mean: read $N$ ($67$ MB).
2. Variance: read $N$ again ($67$ MB).
3. Normalize: read $N$, write $N$ ($134$ MB).

$$
\text{traffic} = 4N \times 4 \text{ B} = 268 \text{ MB}, \qquad T = \frac{2.68\times10^{8}}{2\times10^{11}} = 1.34 \text{ ms}
$$

**(b) Single-pass Welford + fused normalize.** One streaming pass computes mean and $M_2$ simultaneously; a second pass (unavoidable, since normalization needs the finished statistics) reads and writes:

$$
\text{traffic} = 3N \times 4 = 201 \text{ MB}, \qquad T = 1.0 \text{ ms}
$$

If the statistics for a row fit in cache (here a row is $1024 \times 4 = 4$ KB, easily L1-resident) the implementation can process one row completely before moving on, making the second pass a *cache* read rather than a DRAM read:

$$
\text{traffic} = 2N \times 4 = 134 \text{ MB}, \qquad T = 0.67 \text{ ms}
$$

**(c)** Speedup $\approx 2\times$ against the naive version. Flops are essentially unchanged — the win is entirely traffic. **Extra benefit**: Welford avoids the $\overline{x^{2}} - \bar{x}^{2}$ cancellation (Topic 02, Derivation 3.7), whose relative error is amplified by $\mu^{2}/\sigma^{2}$; in fp16/bf16 activations that amplification routinely destroys the variance estimate. Here the fastest algorithm is also the most accurate one.

$$
\boxed{268\,\text{MB} \to 134\,\text{MB} \; (2\times \text{ faster}), \text{ and Welford removes the } \mu^{2}/\sigma^{2} \text{ error amplification}}
$$

*Key takeaway*: normalization layers are pure traffic. This is why every framework fuses them into the neighbouring matmul epilogue, and why the row-blocked ordering matters more than the arithmetic.

### Problem L2.3: Autoregressive decoding is a bandwidth problem

A $7$B-parameter model stored in fp16 generates tokens one at a time on an A100 ($B_{\text{mem}} = 1.55$ TB/s, $P_{\text{peak}} = 312$ Tflop/s). (a) Compute the per-token time and token rate at batch size 1. (b) Compute the arithmetic intensity and confirm the regime. (c) What do int8 quantization, batching, and speculative decoding each change?

**Solution**

**(a)** Every generated token requires reading **all** weights once:

$$
\text{bytes} = 7\times10^{9} \times 2 = 1.4 \times 10^{10} = 14 \text{ GB}
$$

$$
T = \frac{1.4\times10^{10}}{1.55\times10^{12}} = 9.0 \text{ ms} \quad \Longrightarrow \quad \approx 111 \text{ tokens/s}
$$

**(b)** Flops per token $\approx 2 \times 7\times10^{9} = 1.4 \times 10^{10}$ (two flops per parameter):

$$
I = \frac{1.4\times10^{10}}{1.4\times10^{10}} = 1 \; \frac{\text{flop}}{\text{byte}} \quad \ll \quad I^{*} = \frac{312\times10^{12}}{1.55\times10^{12}} \approx 200
$$

Memory-bound by a factor of $200$: the GPU runs at $0.5\%$ of peak. Time spent on arithmetic is $1.4\times10^{10}/3.12\times10^{14} = 45\;\mu$s out of $9$ ms.

**(c)**
- **int8 quantization** halves the bytes again: $7$ GB per token, $4.5$ ms, $\approx 222$ tokens/s. The dequantization arithmetic is *free* (we are $200\times$ below the ridge), which is why weight-only quantization gives near-linear latency wins.
- **Batching** $B$ sequences amortizes the same weight read over $B$ tokens: $I \to B$, so throughput scales nearly linearly until $B \approx I^{*} = 200$, after which the model becomes compute-bound and further batching only trades latency for throughput. (The KV cache grows with $B$ and eventually reclaims bandwidth.)
- **Speculative decoding** has a small draft model propose $k$ tokens which the big model verifies in **one** forward pass — one weight read for up to $k$ accepted tokens, i.e. an artificial batch dimension in the *time* axis. Same principle: raise $I$.

$$
\boxed{T_{\text{token}} = \frac{\text{model bytes}}{B_{\text{mem}}} = 9\,\text{ms} \; (111 \text{ tok/s}), \; I = 1 \ll I^{*} = 200}
$$

*Key takeaway*: LLM inference latency is a division of model size by memory bandwidth. Every serving optimization (quantization, batching, paged KV cache, speculative decoding) is an attack on one of those two numbers.

### Problem L2.4: Batch size and the level-2 to level-3 transition

A linear layer has $W \in \mathbb{R}^{H \times H}$ with $H = 4096$, fp16, on an A100 ($I^{*} \approx 200$ for tensor-core fp16). (a) Compute $I$ for batch size $B$. (b) Find the batch size at which the layer becomes compute-bound. (c) Predict the throughput curve and explain why gradient accumulation with micro-batches is slower than one large batch at equal flop count.

**Solution**

**(a)** For $Y = XW$ with $X \in \mathbb{R}^{B \times H}$: flops $= 2BH^{2}$; bytes $= 2(H^{2} + BH + BH) = 2H^{2} + 4BH$ (fp16 is 2 B/element). For $B \ll H$ the weight read dominates:

$$
I(B) = \frac{2BH^{2}}{2H^{2} + 4BH} \; \approx \; \frac{2BH^{2}}{2H^{2}} = B \quad (B \ll H)
$$

Arithmetic intensity is (to leading order) **the batch size**.

**(b)** Compute-bound requires $I \ge I^{*}$:

$$
B \gtrsim 200
$$

Solving exactly, $\frac{2BH^{2}}{2H^{2}+4BH} = 200$ gives $B(2H^{2} - 800H) = 400H^{2}$, i.e. $B \approx \frac{400H}{2H - 800} \approx 211$ for $H = 4096$.

**(c) Throughput curve.** For $B \lt 200$ time is constant at $T = \text{bytes}/B_{\text{mem}} \approx 2H^{2}/B_{\text{mem}} = 22\;\mu$s regardless of $B$ — so **throughput rises linearly** while latency stays flat. Beyond $B \approx 200$, time grows linearly with $B$ and throughput plateaus at peak.

**Gradient accumulation.** Splitting a batch of $2048$ into $16$ micro-batches of $128$ puts every micro-step at $B = 128 \lt 200$, i.e. in the memory-bound regime where each step costs the same $22\;\mu$s as a single example. Total: $16 \times 22 = 352\;\mu$s versus $2 \times 2048 \times 4096^{2}/P_{\text{peak}} = 220\;\mu$s for one large batch — about $1.6\times$ slower at identical flops and identical mathematics. The weight matrix was re-read from HBM 16 times instead of once.

$$
\boxed{I(B) \approx B; \; \text{compute-bound at } B \gtrsim 200; \; \text{micro-batching re-reads the weights once per micro-step}}
$$

*Key takeaway*: batch size is a roofline parameter before it is a statistical one. "Largest batch that fits" is a hardware statement about reaching $I^{*}$.

### Problem L2.5: Contraction order in linear attention

Linear attention computes $O = Q(K^{\top}V)$ instead of $(QK^{\top})V$, with $Q, K, V \in \mathbb{R}^{N \times d}$. (a) Give the flop count of both orders. (b) Find the crossover. (c) Write both as `einsum` and explain what `optimize=True` does. (d) Why can standard softmax attention *not* use the cheap order?

**Solution**

**(a)**

- $(QK^{\top})V$: first $QK^{\top}$ costs $2N^{2}d$ and materializes an $N \times N$ matrix; then times $V$ costs $2N^{2}d$. Total $4N^{2}d$ flops, $\Theta(N^{2})$ memory.
- $Q(K^{\top}V)$: first $K^{\top}V$ costs $2Nd^{2}$ and produces a $d \times d$ matrix; then $Q$ times it costs $2Nd^{2}$. Total $4Nd^{2}$ flops, $\Theta(d^{2})$ extra memory.

**(b)** Cheaper right-to-left when

$$
4Nd^{2} \lt 4N^{2}d \quad \Longleftrightarrow \quad d \lt N
$$

With $N = 4096$, $d = 64$: $4N^{2}d = 4.3\times10^{9}$ versus $4Nd^{2} = 6.7\times10^{7}$ — a **64-fold** ($N/d$) flop reduction, and the memory drops from $16.8$M entries to $4096$. This linear-in-$N$ scaling is the entire selling point of linear/kernelized attention (Performer, Linear Transformer, and the state-space family).

**(c)**

```python
# both are the same einsum string; only the contraction tree differs
O = np.einsum('nd,md,me->ne', Q, K, V, optimize=True)
print(np.einsum_path('nd,md,me->ne', Q, K, V, optimize='optimal')[1])
```

`optimize=True` searches contraction trees (exhaustively for up to 4 operands, greedily beyond), estimates flops for each, and dispatches each pairwise step to `tensordot` $\to$ `gemm`. Without it, older NumPy evaluates a single nested loop over all indices $n, m, d, e$ at cost $\Theta(N^{2}d^{2})$ — worse than *either* pairwise order.

**(d)** Softmax attention computes $O = \mathrm{softmax}(QK^{\top}/\sqrt{d})V$. The softmax is a **nonlinear, row-coupled** function of the score matrix, so $QK^{\top}$ must exist before $V$ is applied: associativity is destroyed. Linear attention replaces $\exp(q^{\top}k)$ by a factorized kernel $\phi(q)^{\top}\phi(k)$ precisely to restore associativity — the approximation is chosen for the contraction order it enables.

$$
\boxed{(QK^{\top})V: 4N^{2}d; \quad Q(K^{\top}V): 4Nd^{2}; \quad \text{ratio } N/d = 64 \text{ at } N=4096, d=64}
$$

*Key takeaway*: contraction order is an algorithmic choice with asymptotic consequences, and `einsum` will not find it for you unless you pass `optimize=True`.

### Problem L2.6: Benchmarking a kernel honestly

A colleague reports "the new kernel is $3\times$ faster" from this script:

```python
import time
t0 = time.time(); new_kernel(x); print(time.time() - t0)
t0 = time.time(); old_kernel(x); print(time.time() - t0)
```

with `x` of $10^{4}$ float32 elements. List every methodological defect and give a corrected measurement protocol.

**Solution**

**Defects.**

1. **Single measurement.** No estimate of variance; a single sample of a noisy quantity is not a measurement.
2. **No warm-up.** The first call pays page faults on first touch, allocator growth, lazy imports, and (for Numba/JAX/Torch) JIT compilation — often $10^{3}$–$10^{6}\times$ the kernel time.
3. **Cache-state confound.** `new_kernel` runs first on cold cache; `old_kernel` then finds `x` hot in L2. The $10^{4}$ float32 array is $40$ KB — fits in L2, so the second call is measured in a completely different regime.
4. **Timer resolution.** `time.time()` has $\sim 1$–$16$ ms resolution on some platforms; a $10\;\mu$s kernel is unmeasurable. Use `time.perf_counter()`, and repeat inside the timed region.
5. **Dead-code / lazy-evaluation risk.** The result is discarded; a JIT may elide the work, and GPU/async frameworks return before the kernel finishes.
6. **Uncontrolled threading.** BLAS thread counts differ between calls or machines; results are not reproducible.
7. **One size only.** A single $n$ cannot distinguish a constant-overhead win from a bandwidth win; the ranking often flips with $n$.

**Corrected protocol.**

```python
import timeit, numpy as np
from threadpoolctl import threadpool_limits

def bench(fn, x, reps=7, target=0.05):
    fn(x)                                   # warm up: pages, JIT, allocator
    n = 1
    while timeit.timeit(lambda: fn(x), number=n) < target:
        n *= 2                              # amortize timer resolution
    ts = timeit.repeat(lambda: fn(x), number=n, repeat=reps)
    return min(ts) / n                      # min = least-contaminated estimate

with threadpool_limits(limits=1):           # pin threads
    for n in [10**3, 10**5, 10**7, 10**8]:  # sweep sizes across cache levels
        x = np.random.rand(n).astype(np.float32)
        t_new, t_old = bench(new_kernel, x), bench(old_kernel, x)
        gbs = x.nbytes * 2 / t_new / 1e9    # report a RATE, not a time
        print(n, t_old / t_new, gbs)
```

Report `min` (best estimator of uncontended kernel time, since all contamination is non-negative), the spread across repeats, the size sweep, and the achieved GB/s or Gflop/s compared against the roofline — an absolute time alone never says whether further optimization is possible.

$$
\boxed{\text{warm up} \to \text{amortize the timer} \to \text{repeat} \to \text{report min + spread + rate, swept over } n}
$$

*Key takeaway*: a benchmark measures a distribution, not a number, and it must report the *rate* so it can be compared against the hardware limit.

## Level 3 — Challenge

### Problem L3.1: Prove the blocked-matmul traffic bound

For $C = AB$ with all matrices $n \times n$ and fast memory of $M$ words, (a) derive the traffic of the $b$-blocked algorithm exactly; (b) minimize over $b$ subject to the capacity constraint; (c) compare with the naive triple loop and with the Hong–Kung lower bound; (d) state what changes when only $C$ must be resident.

**Solution**

**(a) Exact count.** Tiles: $t = n/b$ per dimension. The algorithm is

$$
\text{for } I, J: \quad \text{load } C_{IJ}; \quad \text{for } K: \; \text{load } A_{IK}, B_{KJ}, \; C_{IJ} \mathrel{+}= A_{IK}B_{KJ}; \quad \text{store } C_{IJ}
$$

Words moved:

- $C$: loaded and stored once per output tile: $2 t^{2} b^{2} = 2n^{2}$.
- $A$: tile $A_{IK}$ is loaded once for each $J$: $t^{3}b^{2} = n^{3}/b$.
- $B$: symmetrically $n^{3}/b$.

$$
Q(b) = 2n^{2} + \frac{2n^{3}}{b}
$$

**(b) Optimize.** $Q$ is strictly decreasing in $b$, so take the largest $b$ permitted by residency of one tile from each of $A$, $B$, $C$:

$$
3b^{2} \le M \quad \Longrightarrow \quad b^{*} = \left\lfloor \sqrt{M/3} \right\rfloor, \qquad Q^{*} = 2n^{2} + 2\sqrt{3}\,\frac{n^{3}}{\sqrt{M}}
$$

**(c) Comparison.**

- **Naive triple loop** ($C_{ij} \mathrel{+}= \sum_k A_{ik}B_{kj}$ with $n^{2} \gg M$): the inner loop streams a row of $A$ and a *column* of $B$; the column is evicted before reuse, giving $Q_{\text{naive}} = \Theta(n^{3})$ — a factor $b^{*} = \Theta(\sqrt{M})$ worse.
- **Hong–Kung lower bound**: $Q \ge \Omega(n^{3}/\sqrt{M})$ for any schedule of the classical algorithm (proved by the red–blue pebble game: any $M$-word window of fast memory can host at most $\Theta(M^{3/2})$ useful multiply–adds, by Loomis–Whitney applied to the three projections of the $n \times n \times n$ index cube). The blocked algorithm matches it up to the constant $2\sqrt{3}$, hence is **communication-optimal**.

**(d) Weaker residency requirement.** If only the $C$ tile must stay resident while $A$- and $B$-*panels* stream through (the Goto layout: an $m_c \times k_c$ packed panel of $A$ in L2, a $k_c \times n$ panel of $B$ in L3), the constraint becomes $b^{2} + 2bk \le M$ with $k$ the panel depth, admitting larger $b$ for the same $M$ — which is exactly why real BLAS uses rectangular, level-specific blocking rather than one square tile. The asymptotic $\Theta(n^{3}/\sqrt{M})$ is unchanged; only the constant improves, and it is the constant that separates $60\%$ of peak from $92\%$.

$$
\boxed{Q(b) = 2n^{2} + \frac{2n^{3}}{b}, \quad b^{*} = \sqrt{M/3}, \quad Q^{*} = \Theta\!\left( \frac{n^{3}}{\sqrt{M}} \right) \text{ — optimal}}
$$

*Key takeaway*: the $\sqrt{M}$ is a theorem, not a heuristic. Cache capacity buys reuse only at a square-root rate, which is why the memory wall keeps rising even as caches grow.

### Problem L3.2: Roofline analysis of a transformer block

For a transformer block with hidden size $H$, sequence length $N$, batch $B$, MLP expansion $4H$, in fp16 on an A100 ($I^{*} \approx 200$): (a) count flops and weight bytes per block; (b) compute $I$ for training (large $BN$) and for single-token decoding; (c) find the token count at which the attention score matrix dominates traffic; (d) rank the optimizations.

**Solution**

**(a) Per block.** Weights: $4H^{2}$ (Q, K, V, O projections) $+\,8H^{2}$ (MLP up and down) $= 12H^{2}$ parameters, i.e. $24H^{2}$ bytes in fp16. Flops for $T = BN$ tokens:

$$
F_{\text{proj+MLP}} = 2 \times 12H^{2} \times T = 24H^{2}T, \qquad F_{\text{attn}} = 4BN^{2}H \; (\text{scores} + \text{values})
$$

**(b) Intensity.**

*Training / prefill* ($T$ large): weights are read once per block and amortized over all $T$ tokens; activations dominate traffic at $\Theta(TH)$ elements:

$$
I \approx \frac{24H^{2}T}{24H^{2} + 4TH \cdot(\text{a few})} \; \xrightarrow[T \gg H]{} \; \Theta(H) \gg I^{*}
$$

**compute-bound** — the intended operating point.

*Decoding* ($T = 1$): the same $24H^{2}$ bytes are read for $24H^{2}$ flops:

$$
I \approx 1 \; \ll \; I^{*} = 200
$$

**memory-bound by $200\times$**, matching Problem L2.3.

**(c) When attention traffic dominates.** Projections/MLP traffic per block is $\Theta(H^{2})$ (weights) $+\,\Theta(TH)$ (activations); the materialized score matrix costs $\Theta(BN^{2})$ elements. Attention dominates when

$$
BN^{2} \gtrsim BNH \quad \Longleftrightarrow \quad N \gtrsim H
$$

For $H = 4096$, that is sequences beyond $\approx 4096$ tokens — precisely where long-context training becomes attention-traffic-limited and FlashAttention's $\Theta(N^{2}d^{2}/M)$ accounting (Problem L3.4) becomes essential. Note the crossover for *flops* is the same $N \approx H$, but the traffic crossover bites first because the score matrix is written and re-read while the matmuls are blocked.

**(d) Ranking.**

| Regime | Binding resource | Highest-value optimizations |
|---|---|---|
| Training, $N \lt H$ | flops | tensor cores, bf16/fp8, larger batch, fused epilogues |
| Training, $N \gt H$ | attention traffic | FlashAttention, sparse/sliding-window attention, sequence parallelism |
| Decoding, $B$ small | weight bandwidth | weight quantization (int8/int4), batching, speculative decoding |
| Decoding, $B$ large | KV-cache bandwidth | paged/grouped KV (MQA/GQA), KV quantization, cache eviction |

$$
\boxed{I_{\text{train}} = \Theta(H) \gg I^{*}; \; I_{\text{decode}} \approx 1 \ll I^{*}; \; \text{attention traffic dominates once } N \gtrsim H}
$$

*Key takeaway*: one architecture occupies three different roofline regimes depending on $B$, $N$, and phase — which is why "make the model faster" has no single answer without naming the regime.

### Problem L3.3: Why `min` and not `mean`

Model a timing measurement as $T_i = \tau + \varepsilon_i$ with $\tau$ the true kernel time and $\varepsilon_i \ge 0$ i.i.d. contamination. (a) Show $\min_i T_i$ is a consistent estimator of $\tau$ under a mild condition while $\bar{T}$ is biased. (b) Quantify the bias with $\varepsilon \sim \mathrm{Exp}(\lambda)$. (c) Give the case where reporting the mean is correct. (d) Why does this argument fail for cache-sensitive benchmarks?

**Solution**

**(a)** Assume $\varepsilon \ge 0$ with $\Pr[\varepsilon \lt \delta] \gt 0$ for every $\delta \gt 0$ (contamination can be arbitrarily small). Then for any $\delta \gt 0$,

$$
\Pr\!\left[ \min_{i \le n} T_i \gt \tau + \delta \right] = \left( \Pr[\varepsilon \gt \delta] \right)^{n} = (1 - p_\delta)^{n} \to 0
$$

since $p_\delta \gt 0$. And $\min_i T_i \ge \tau$ always. Hence $\min_i T_i \to \tau$ in probability: **consistent**. Meanwhile

$$
\mathbb{E}[\bar{T}] = \tau + \mathbb{E}[\varepsilon] \gt \tau \quad \text{for all } n
$$

— the mean converges to the *wrong* quantity, and $\mathbb{E}[\varepsilon]$ depends on machine load, not on the kernel.

**(b)** With $\varepsilon \sim \mathrm{Exp}(\lambda)$, $\mathbb{E}[\varepsilon] = 1/\lambda$, so the mean's bias is $1/\lambda$ regardless of $n$. The minimum of $n$ exponentials is $\mathrm{Exp}(n\lambda)$:

$$
\mathbb{E}\!\left[ \min_i T_i \right] = \tau + \frac{1}{n\lambda}
$$

Bias shrinks as $1/n$. With $\tau = 1$ ms and $1/\lambda = 0.2$ ms, the mean over-reports by $20\%$ forever, while $7$ repeats bring the minimum's bias to $2.9\%$ and $50$ repeats to $0.4\%$.

**(c)** When the quantity of interest *is* the contaminated one: production latency SLOs, tail percentiles (p99), throughput under realistic multi-tenancy, or any comparison meant to predict user-visible behaviour. Then report the full distribution — median and p95/p99 — because contention is part of the product, not noise.

**(d)** The derivation assumes contamination is additive and non-negative *relative to a fixed $\tau$*. Repetition changes $\tau$ itself when it changes the cache state: run 1 is cold, runs $2, \dots, n$ are hot, so $\min$ estimates the **hot-cache** kernel time, which can be $10\times$ below the cold-cache time the application will actually experience. The fix is to decide the regime explicitly — flush caches or rotate through several buffers for cold measurements, or state plainly that the number is a hot-cache figure.

$$
\boxed{\mathbb{E}[\min] = \tau + \frac{1}{n\lambda} \to \tau, \qquad \mathbb{E}[\bar{T}] = \tau + \frac{1}{\lambda} \; \text{(biased for all } n\text{)}}
$$

*Key takeaway*: `min` estimates the kernel; percentiles estimate the service. Report the one that matches the question, and always say which cache state you measured.

### Problem L3.4: FlashAttention traffic accounting

Standard attention materializes the $N \times N$ score matrix in HBM; FlashAttention tiles it into on-chip SRAM of $M$ elements and fuses the softmax. With head dimension $d$: (a) count HBM traffic for both; (b) derive the tile sizes from the capacity constraint; (c) compute the ratio for $N = 4096$, $d = 64$, $M = 10^{5}$; (d) explain the numerical mechanism that makes tiling possible and why the backward pass recomputes.

**Solution**

**(a) Standard.** Write $S = QK^{\top}/\sqrt{d}$ ($N^{2}$), read it for the softmax ($N^{2}$), write $P$ ($N^{2}$), read $P$ for $PV$ ($N^{2}$), plus $O(Nd)$ for the inputs and output:

$$
Q_{\text{std}} = \Theta\!\left( 4N^{2} + Nd \right) \text{ elements}
$$

**FlashAttention.** Outer loop over $T_c = N/B_c$ blocks of $(K, V)$; for each, the *entire* $Q$ and the running output $O$ are streamed:

$$
Q_{\text{flash}} = \Theta\!\left( T_c \cdot Nd \right) = \Theta\!\left( \frac{N^{2}d}{B_c} \right)
$$

**(b) Capacity.** SRAM must hold a $K$-block, a $V$-block, a $Q$-block, and the score tile. With $B_c = \Theta(M/d)$ column-block size and $B_r = \min(\Theta(M/d), d)$ row-block size, the four residents total $\Theta(M)$. Substituting $B_c = M/(4d)$:

$$
Q_{\text{flash}} = \Theta\!\left( \frac{N^{2}d}{M/(4d)} \right) = \Theta\!\left( \frac{N^{2}d^{2}}{M} \right)
$$

**(c) Ratio.**

$$
\frac{Q_{\text{std}}}{Q_{\text{flash}}} \approx \frac{4N^{2}}{4N^{2}d^{2}/M} = \frac{M}{d^{2}} = \frac{10^{5}}{4096} \approx 24\times
$$

fewer HBM accesses, while the **flop count is unchanged** (indeed slightly higher because of recomputation). Measured end-to-end speedups of $2$–$4\times$ are smaller than $24\times$ because the matmuls themselves were already efficient and only the attention traffic is removed — Amdahl, applied to bytes.

**(d) The numerical mechanism.** Softmax appears to require a global maximum and a global sum over each row, which would forbid tiling. The **online softmax** update supplies both incrementally: for a new block with local max $m^{\text{new}}$ and local sum $\ell^{\text{new}}$,

$$
m \leftarrow \max(m, m^{\text{new}}), \qquad \ell \leftarrow e^{m_{\text{old}} - m}\ell + e^{m^{\text{new}} - m}\ell^{\text{new}}, \qquad O \leftarrow e^{m_{\text{old}} - m} O + e^{m^{\text{new}} - m} O^{\text{new}}
$$

Every exponent is $\le 0$, so nothing overflows — the same max-subtraction that makes log-sum-exp stable (Topic 02, and Topic 05's Derivation of the log-sum-exp bound), now in *streaming* form. This is a rare and instructive alignment: the rescaling that makes the algorithm numerically stable is exactly what makes it tileable.

**Backward recomputation.** Storing $P$ for the backward pass would reinstate the $\Theta(N^{2})$ traffic that tiling removed. Instead the backward pass keeps only the row statistics $(m, \ell)$ — $\Theta(N)$ memory — and *recomputes* each score tile in SRAM. Extra flops, far less traffic: the correct trade on the memory-bound side of the roofline, and the same reasoning as gradient checkpointing.

$$
\boxed{Q_{\text{std}} = \Theta(N^{2}), \; Q_{\text{flash}} = \Theta\!\left( \frac{N^{2}d^{2}}{M} \right), \; \text{ratio} = \frac{M}{d^{2}} \approx 24 \text{ at } N=4096, d=64}
$$

*Key takeaway*: FlashAttention is Theorem 2.7's cache blocking plus Topic 02's max-subtraction trick. No new mathematics — two classical ideas composed, worth several billion dollars of saved compute.